# 01 - FSL FAST + MNI alignment of the 20 HCP healthy scans

**Just press Run All.** Needs: Accelerator = None (CPU), Internet = On, the private dataset `hcp-t2-healthy-20` attached, and a Kaggle Secret named `GITHUB_TOKEN`.

What it does: installs FSL -> downloads our code from GitHub -> aligns each scan to MNI space, runs FAST, builds the 4 tissue classes (CSF, GM, WM, dark) -> shows pictures and a summary table -> zips the results.
With `LIMIT = 1` (default) it processes one subject (~6 min); set `LIMIT = None` for all 20 (~1-2 hours). The first cell that fails stops the run with a clear message.

## 0. Settings

In [ ]:
LIMIT = 1          # number of subjects to process: 1 = quick test (~6 min), None = all 20 (~1-2 h)
CLEAN = True        # True: delete old results first (needed after any change to the label definition)
WORKERS = None      # None = automatic (CPU cores - 1, at most 3)
DARK_FRAC = 0.5     # 'dark' class = brain voxels below this fraction of the subject's median white-matter T2

## 1. Install FSL (FAST, FLIRT, MNI templates) - a few minutes

In [ ]:
import os, subprocess, sys
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd[:80]}")
    return r.returncode

sh("mkdir -p /tmp/mm && cd /tmp/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba", check=True)
sh("/tmp/mm/bin/micromamba create -y -p /tmp/fsl-env -c https://fsl.fmrib.ox.ac.uk/fsldownloads/fslconda/public/ -c conda-forge fsl-fast4 fsl-flirt fsl-data_standard", check=True)

os.environ["FSLDIR"] = "/tmp/fsl-env"
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"
os.environ["PATH"] = "/tmp/fsl-env/bin:" + os.environ["PATH"]

In [ ]:
import shutil
assert shutil.which("fast") and shutil.which("flirt"), "FSL tools not found: the install above failed"
MNI = "/tmp/fsl-env/data/standard/MNI152_T1_1mm_brain.nii.gz"
assert os.path.exists(MNI), f"MNI template missing: {MNI}"
print("FSL OK:", shutil.which("fast"), shutil.which("flirt"), "| template OK")

## 2. Get our code from the private GitHub repo (needs the `GITHUB_TOKEN` secret)

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/repo"
sh(f"cd /kaggle/working && rm -rf repo && git clone -q https://{token}@github.com/PiyushGarg0146/lf-synthseg-btp.git repo", check=True)
sh(f"{sys.executable} -m pip install -q nibabel scipy matplotlib pyyaml scikit-learn")
sh(f"cd {REPO} && git log --oneline | head -3 && ls", check=True)

## 3. Run the pipeline on all 20 scans

In [ ]:
HCP = "/kaggle/input"
OUT = "/kaggle/working/hcp_processed"
if CLEAN:
    sh(f"rm -rf {OUT}")
cores = os.cpu_count() or 1
workers = WORKERS or max(1, min(3, cores - 1))
print(f"CPU cores: {cores}, workers: {workers}")
cmd = f"cd {REPO} && {sys.executable} -u scripts/fsl_pipeline.py --src {HCP} --dst {OUT} --workers {workers} --dark-frac {DARK_FRAC} --resume" + (f" --limit {LIMIT}" if LIMIT else "")
# stream the output live so progress is visible
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
assert p.returncode == 0, "pipeline failed - see the messages above"

## 4. Check the results

In [ ]:
import glob, numpy as np, nibabel as nib
files = sorted(glob.glob(OUT + "/*/labels.nii.gz"))
print(len(files), "subjects finished (expected:", LIMIT or 20, ")")
print("subject | mean T2 CSF, GM, WM, dark | dark share of brain | share of dark voxels that are bright (T2>0.6)")
for f in files:
    d = f.rsplit("/", 1)[0]
    t = np.asarray(nib.load(d + "/t2.nii.gz").dataobj); l = np.asarray(nib.load(f).dataobj)
    means = [round(float(t[l == k].mean()), 3) for k in (1, 2, 3, 4)]
    brain = (l > 0).sum()
    print(d.rsplit("/", 1)[1], means, round(float((l == 4).sum() / brain), 3), round(float((t[l == 4] > 0.6).mean()), 3))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
pngs = sorted(glob.glob(OUT + "/*/qc.png"))
cols = 3; rows = (len(pngs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, 4.2 * rows))
for ax in np.ravel(axes): ax.axis("off")
for ax, pth in zip(np.ravel(axes), pngs):
    ax.imshow(mpimg.imread(pth)); ax.set_title(pth.split("/")[-2])
plt.tight_layout(); plt.show()

## 5. Package results for download (private - HCP data terms do not allow sharing)

In [ ]:
sh(f"cd /kaggle/working && rm -f hcp_processed/*/native_ras1mm.nii.gz hcp_processed/*/t2_mni_raw.nii.gz && rm -f hcp_processed.zip && zip -qr hcp_processed.zip hcp_processed && ls -lh hcp_processed.zip", check=True)
print("Download hcp_processed.zip from the Output panel (right side) -> /kaggle/working")